# ALIM + Qualcomm VLM — Colab Integration

**Architecture (unchanged, enforced structurally in this notebook):**

```
real datasheet PDF -> render page -> Qualcomm-catalog VLM -> structured JSON evidence
                                                                    |
                                                                    v
                                          alim.decision.core.extract_parameter
                                          (the EXISTING, unmodified decision core)
                                                                    |
                                                                    v
                                        parameter / value / unit / condition / page
```

The VLM never decides between candidates and is never allowed to invent a
missing field — see `alim/integrations/qualcomm/schema.py` for the exact
evidence contract it's asked to fill in. This notebook does not
reimplement any decision logic; it only wires a model backend into the
existing `alim.integrations.qualcomm.provider.QualcommVLMProvider`.

## Honest scope of what this notebook can and cannot do

- **Colab has no Snapdragon silicon.** There is no Hexagon NPU, no
  GenieX/QAIRT runtime available in a standard Colab VM. Running the model
  here means standard GPU inference via Hugging Face `transformers` —
  useful for validating the prompt, the evidence schema, and the ALIM
  integration end-to-end with a real vision-language model, but it is
  **not** a measurement of on-device Snapdragon performance.
- **Real, on-device Snapdragon numbers already exist**, published by
  Qualcomm for this exact model (see `docs/QUALCOMM.md` for the citation
  and the full table) — this notebook does not re-derive those numbers,
  it cites them.
- **Getting genuine on-device performance evidence** for your own prompt/
  workload requires either (a) a real Snapdragon-on-Windows device running
  GenieX, or (b) Qualcomm AI Hub Workbench's hosted device farm via the
  `qai-hub` Python SDK — which *can* be called from this Colab notebook,
  but requires **your own** Qualcomm AI Hub account and API token. That
  section is included below, clearly marked, and was not run by the
  assistant that built this notebook (no such account/token available in
  that environment).
- **Model selection:** `Qwen3-VL-4B-Instruct` was selected over the
  originally-suggested `Qwen2.5-VL-7B-Instruct` after checking the current
  Qualcomm AI Hub catalog directly — same supported-chipset list
  (Snapdragon X Elite / X Plus 8-Core / X2 Elite), same runtime path
  (GenieX–QAIRT), same Apache-2.0 license, roughly half the parameters
  (better on-device latency/memory), newer-generation architecture. See
  `docs/QUALCOMM.md` for the full comparison and citations.

## MOCK_MODE

Set `MOCK_MODE = True` to run this entire notebook immediately with a
fixture model backend (no GPU, no download, no wait) — this validates
every cell except the actual model call. Set `MOCK_MODE = False` once you
want to run a real Qwen3-VL-4B-Instruct inference call on Colab's GPU.


In [ ]:
MOCK_MODE = True  # flip to False once you're ready to load the real model
QUERY = "operating supply voltage"
PDF_NAME = "nRF24L01P.PDF"     # the real, published Nordic nRF24L01+ datasheet
TARGET_PAGE = 12               # the Absolute-Maximum-Ratings page with the mislabeled header


## 1. Install dependencies

Only what's actually needed. `transformers`/`accelerate`/`torch` are only required when `MOCK_MODE = False`.

In [ ]:
!pip install -q pymupdf pillow
if not MOCK_MODE:
    !pip install -q "transformers>=4.46" accelerate torch --extra-index-url https://download.pytorch.org/whl/cu121


## 2. Install the ALIM engine

Replace `ALIM_REPO_URL` with the URL you pushed this repository to. If you've instead uploaded/mounted the repo directly in this Colab session, skip the clone and just `%cd` into it.

In [ ]:
ALIM_REPO_URL = "<paste your pushed alim repository URL here>"

import os
if not os.path.exists("alim_engine_repo"):
    !git clone {ALIM_REPO_URL} alim_engine_repo
%cd alim_engine_repo
!pip install -q -e .
!python fetch_fixtures.py


## 3. Render the target datasheet page

Uses the same PyMuPDF rendering path as `alim.integrations.qualcomm.provider.QualcommVLMProvider._render_page` -- this cell just makes the image visible for you to sanity-check before sending it to a model.

In [ ]:
import fitz  # PyMuPDF

pdf_path = f"alim/tests/fixtures/{PDF_NAME}"
doc = fitz.open(pdf_path)
page = doc[TARGET_PAGE - 1]
pix = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0))
image_bytes = pix.tobytes("png")

from IPython.display import Image, display
display(Image(data=image_bytes))


## 4. The evidence prompt (unchanged, imported from the ALIM repo)

In [ ]:
from alim.integrations.qualcomm.schema import build_prompt

prompt = build_prompt(QUERY)
print(prompt)


## 5. Model backend

Two backends, selected by `MOCK_MODE`. Both expose the same signature the
`QualcommVLMProvider` contract expects: `generate_fn(image_bytes, prompt) -> raw_text`.

**Mock backend**: returns realistic evidence for the real nRF24L01+ page
this notebook defaults to — including the actual mislabeled-header
situation (the table's own header says "Operating conditions" even though
the section is Absolute Maximum Ratings) so you can see ALIM's negative
selection handle it correctly before waiting on a real model.

**Real backend**: loads `Qwen/Qwen3-VL-4B-Instruct` via `transformers` on
Colab's GPU. This is the *original* (non-Qualcomm-optimized) checkpoint --
the `qualcomm/Qwen3-VL-4B-Instruct` Hugging Face repo only hosts
pre-exported on-device runtime assets (GenieX/GENIE packages, a GGUF for
llama.cpp), not a standard `transformers`-loadable checkpoint. Loading
code below uses the generic `AutoModelForImageTextToText` class; if your
installed `transformers` version predates support for this architecture,
upgrade (`pip install -U transformers`) or check the model card at
https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct for the current class
name — this was not executed or verified against a live download in the
environment that built this notebook (no network route to huggingface.co
there).


In [ ]:
def mock_generate_fn(image_bytes, prompt):
    # Realistic evidence for the real nRF24L01+ Absolute Maximum Ratings
    # page (values read directly off the real datasheet earlier in this
    # project). Deliberately reproduces the real mislabeled-header
    # situation: the table's own header says "Operating conditions" but
    # the caption below it correctly says "Absolute maximum ratings".
    import json
    return json.dumps([
        {"section": "Absolute maximum ratings", "table_caption": "Table 2. Absolute maximum ratings",
         "parameter": "VDD", "symbol": "VDD", "condition": None, "min": "-0.3", "max": "3.6", "unit": "V",
         "page": TARGET_PAGE, "uncertain": False},
    ])

def real_generate_fn(image_bytes, prompt):
    import torch
    from transformers import AutoModelForImageTextToText, AutoProcessor
    from PIL import Image
    import io

    model_id = "Qwen/Qwen3-VL-4B-Instruct"
    if not hasattr(real_generate_fn, "_model"):
        real_generate_fn._processor = AutoProcessor.from_pretrained(model_id)
        real_generate_fn._model = AutoModelForImageTextToText.from_pretrained(
            model_id, torch_dtype=torch.bfloat16, device_map="auto")

    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text_prompt = real_generate_fn._processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = real_generate_fn._processor(text=[text_prompt], images=[image], return_tensors="pt").to(
        real_generate_fn._model.device)
    output_ids = real_generate_fn._model.generate(**inputs, max_new_tokens=1024)
    return real_generate_fn._processor.decode(output_ids[0][inputs["input_ids"].shape[1]:],
                                                skip_special_tokens=True)

generate_fn = mock_generate_fn if MOCK_MODE else real_generate_fn


## 6. Run perception -> ALIM decision on ONE page (unchanged decision core)

This deliberately feeds ALIM only the Absolute-Maximum-Ratings page's evidence, in isolation, to show negative selection reject it on its own merits. With no competing Operating-table evidence in scope yet, the correct result here is **`NOT_FOUND`** (the only candidate was correctly rejected, nothing else was offered) -- not a bug. §7 below runs the real multi-page pipeline and gets the actual answer.

In [ ]:
from alim.integrations.qualcomm.provider import QualcommVLMProvider
from alim.decision.core import extract_parameter

provider = QualcommVLMProvider(generate_fn=generate_fn)
candidates = provider.perceive(pdf_path, page_number=TARGET_PAGE, query=QUERY)

print("VLM evidence:")
for c in candidates:
    print(f"  parameter={c.detected_parameter!r} symbol={c.symbol!r} "
          f"min={c.min_val} max={c.max_val} unit={c.unit} "
          f"section={c.section!r} confidence={c.section_confidence}")

decision = extract_parameter(candidates, QUERY)
print("\nALIM decision:", decision.status)
for c in decision.accepted:
    print(f"  ACCEPTED: {c.detected_parameter} = {c.min_val}\u2013{c.max_val} {c.unit} "
          f"(condition: {c.condition or '-'}, page {c.page})")
for c, reason in decision.rejected:
    print(f"  rejected: {c.detected_parameter} = {c.min_val}\u2013{c.max_val} -> {reason}")


## 7. Full ALIM pipeline entrypoint (structural pass + VLM fallback together)

This is the actual top-level engine call — `alim.api.extract.extract()` —
which runs the structural pass first and only escalates to the VLM
provider for pages whose table schema it could not confidently classify,
and only if the structural pass alone didn't already produce a confident
answer. This is the same two-phase orchestration used throughout the
rest of the ALIM test suite, exercised here with a real VLM backend
instead of a fixture.


In [ ]:
from alim.api.extract import extract

result = extract(pdf_path, QUERY, vlm_provider=provider, trace=True)
import json
print(json.dumps(result, indent=2))


## 8. (Optional) Real Snapdragon device profiling via Qualcomm AI Hub Workbench

Requires your own Qualcomm AI Hub account and API token — sign up at
https://myaccount.qualcomm.com/signup, then get your token from
https://aihub.qualcomm.com (Workbench). **Not run in the environment that
built this notebook** — no such account/token was available there. This
is the officially-documented path to get genuine physical-device latency
and memory numbers rather than Colab GPU numbers, without needing to own
a Snapdragon device yourself.


In [ ]:
# !pip install -q qai-hub
# !qai-hub configure --api_token YOUR_TOKEN_HERE
#
# import qai_hub as hub
# device = hub.Device("Snapdragon X Elite CRD")
# # See https://github.com/qualcomm/ai-hub-models/tree/main/qai_hub_models/models/qwen3_vl_4b_instruct
# # for the model-specific compile/profile job submission code -- this varies
# # by model and was not executed here.


## 9. What this notebook does and does not prove

**Proves (once you run it with `MOCK_MODE = False` on a real GPU):** the
evidence schema is fillable by a real current-generation VLM, the JSON
parsing is robust to that model's actual output format, and the existing,
unmodified ALIM decision core correctly resolves the real
mislabeled-header adversarial case using real model output rather than a
fixture.

**Does not prove:** on-device Snapdragon latency/memory (see §8 for the
real path to that), that Qwen3-VL-4B-Instruct is more *accurate* than
Qwen2.5-VL-7B-Instruct at this specific datasheet-reading task
specifically (both are plausible; only real evaluation on your adversarial
suite settles this), or that this generalizes beyond the datasheets in
the existing ALIM fixture set.


## 10. Batch test across the full real-datasheet corpus (22 documents, 8+ device categories)

Runs the same `extract()` call across every fixture in
`alim/tests/fixtures/` with a representative query per document,
using whichever `generate_fn` you configured above (mock or real).

**A structural-only baseline (no VLM) was already run and recorded** --
see `docs/QUALCOMM.md` §6 for the exact numbers. Of 22 real documents:
only 4 resolved with a confident structural answer, 10 came back
`SCHEMA_UNKNOWN` (a table-shaped grid was found but its schema wasn't
recognized -- these are the documents where the VLM fallback actually
matters), 7 came back `NOT_FOUND` (structural candidates existed on some
of these but didn't match the query terms -- a vocabulary gap, not
necessarily a VLM problem), and 1 came back `AMBIGUOUS_MISSING_CONDITION`
(LM7805 -- multiple input-voltage values with no distinguishing condition
text, a genuinely interesting real case). Run this cell with a real VLM
to see how many of the 10 `SCHEMA_UNKNOWN` cases it actually resolves.


In [ ]:
import time

QUERIES_BY_FILE = {
    "DS18b20.pdf": "supply voltage", "LM35.pdf": "supply voltage", "MQ-7.pdf": "supply voltage",
    "nRF24L01P.PDF": "operating supply voltage", "MLX90614.pdf": "external supply",
    "LM317.pdf": "operating input to output differential voltage", "LM7805.pdf": "input voltage",
    "LM358.pdf": "supply voltage", "LM741.pdf": "supply voltage", "TL082.pdf": "supply voltage",
    "LM339.pdf": "supply voltage", "NE555.pdf": "supply voltage", "CA3306.pdf": "supply voltage",
    "MCP4725.pdf": "supply voltage", "CD4017.pdf": "supply voltage", "KA34063.pdf": "supply voltage",
    "BMP180.pdf": "supply voltage", "VL53L0X.pdf": "supply voltage", "STM32F103C8.pdf": "supply voltage",
    "AT24C02A.pdf": "supply voltage", "AT45DB041B.pdf": "supply voltage", "74HC595.pdf": "supply voltage",
}

results = []
for fname, q in QUERIES_BY_FILE.items():
    path = f"alim/tests/fixtures/{fname}"
    batch_provider = QualcommVLMProvider(generate_fn=generate_fn)
    t0 = time.time()
    try:
        r = extract(path, q, vlm_provider=batch_provider)
        status = r["status"]
    except Exception as e:
        status = f"ERROR: {type(e).__name__}"
    dt = time.time() - t0
    n_vlm_calls = len(r.get("vlm_calls", [])) if isinstance(r, dict) else 0
    results.append((fname, q, status, dt, n_vlm_calls))
    print(f"{fname:18s} {q:45s} {status:26s} {dt:5.1f}s  vlm_calls={n_vlm_calls}")

print("\n--- summary ---")
from collections import Counter
print(Counter(s for _, _, s, _, _ in results))
print(f"total VLM calls made: {sum(n for *_, n in results)}")
print(f"total time: {sum(d for *_, d, _ in results):.1f}s")


**Compare this table against the structural-only baseline in
`docs/QUALCOMM.md` §6.** The interesting numbers: how many of the 10
`SCHEMA_UNKNOWN` documents flipped to a real answer, whether any new
`AMBIGUOUS`/`AMBIGUOUS_MISSING_CONDITION` cases appeared (a sign the VLM
surfaced a genuine multi-candidate situation the structural pass never
saw), and total VLM calls/latency for the batch -- this is the real
cost number for evaluating whether this is practical at the scale you
need.
